# 레슨 06 — 필터, 정렬, 파생 컬럼 정답지

> 교사·관리자 전용. 학생에게 배포하지 않는다.

이 정답지는 학생용 `mission.md` 의 문제 1~15와 번호가 1:1로 대응한다. 이번 강의의 핵심은 원본 성적표에서 운영에 필요한 파생 지표를 만들고, 조건과 순위로 학생군을 해석하는 것이다.

## 환경 셀

In [ ]:
import os
import pandas as pd
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/06/data"
else:
    DATA_BASE = "./data"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)
print("data base:", DATA_BASE)

---

## 문제 1 정답 — 성적표 데이터 불러오기

In [ ]:
df = pd.read_csv(f"{DATA_BASE}/student_performance.csv")

print("shape:", df.shape)
print("columns:", list(df.columns))
print("dtypes:")
print(df.dtypes)
print("앞 5행:")
print(df.head())
print("뒤 3행:")
print(df.tail(3))
print("결측치:")
print(df.isna().sum())

print(f"학생 수는 {len(df):,}명이고, 점수 과목은 math, english, science 3개입니다.")

### 왜 이 코드가 정답인지

분석 전에 데이터의 크기, 열 이름, 타입, 결측 여부를 확인해야 한다. 이 성적표는 학생 480명과 9개 열로 구성되어 있으며, 점수 열은 수학, 영어, 과학 세 개다. 결측이 없다는 사실을 확인하면 이후 필터와 파생 컬럼 계산에서 결측 처리 없이 진행할 수 있다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 행 수 | 480 |
| 열 수 | 9 |
| 결측치 | 0개 |

---

## 문제 2 정답 — 점수 열 기본 통계

In [ ]:
score_cols = ["math", "english", "science"]

print(df[score_cols].describe())

subject_mean = df[score_cols].mean()
subject_max = df[score_cols].max()
subject_min = df[score_cols].min()

print("과목별 평균:")
print(subject_mean)
print("과목별 최고점:")
print(subject_max)
print("과목별 최저점:")
print(subject_min)
print("평균이 가장 높은 과목:", subject_mean.idxmax())

### 왜 이 코드가 정답인지

`score_cols` 로 점수 열을 묶어 두면 여러 과목에 같은 연산을 반복하기 쉽다. `describe()` 는 평균, 표준편차, 사분위수, 최솟값, 최댓값을 한 번에 보여준다. `idxmax()` 는 평균이 가장 큰 열 이름을 반환하므로 평균이 가장 높은 과목을 찾는 데 적합하다.

**예상 핵심값**

```text
math 평균 약 85.07
english 평균 약 83.83
science 평균 약 85.88
평균 1위 과목: science
```

---

## 문제 3 정답 — 평균 점수와 총점 만들기

In [ ]:
df["avg_score"] = df[score_cols].mean(axis=1)
df["total_score"] = df[score_cols].sum(axis=1)

print(df[["student_id", "math", "english", "science", "avg_score", "total_score"]].head())

print("평균 점수 평균:", f"{df['avg_score'].mean():.2f}")
print("평균 점수 중앙값:", f"{df['avg_score'].median():.2f}")
print("평균 점수 최고:", f"{df['avg_score'].max():.2f}")
print("평균 점수 최저:", f"{df['avg_score'].min():.2f}")

print("평균 90점 이상:", (df["avg_score"] >= 90).sum())
print("평균 75점 미만:", (df["avg_score"] < 75).sum())

### 왜 이 코드가 정답인지

학생별 평균과 총점은 한 행 안의 세 과목을 계산해야 하므로 `axis=1` 을 사용한다. `avg_score` 는 과목 수가 바뀌어도 비교하기 쉬운 지표이고, `total_score` 는 전체 점수 규모를 보여준다. 평균 90점 이상과 75점 미만 학생 수를 세면 우수군과 지원 필요군의 대략적인 크기를 알 수 있다.

**예상 핵심값**

```text
평균 90점 이상: 107명
평균 75점 미만: 26명
```

---

## 문제 4 정답 — 우수 학생과 지원 필요 학생 필터

In [ ]:
high_achievers = df[df["avg_score"] >= 90]
low_avg_students = df[df["avg_score"] < 75]
low_attendance = df[df["attendance_rate"] < 0.8]
low_homework = df[df["homework_rate"] < 0.6]

print("평균 90점 이상:", len(high_achievers))
print(high_achievers.head())
print("평균 75점 미만:", len(low_avg_students))
print(low_avg_students.head())
print("출석률 0.8 미만:", len(low_attendance))
print(low_attendance.head())
print("과제 제출률 0.6 미만:", len(low_homework))
print(low_homework.head())

### 왜 이 코드가 정답인지

조건 필터는 특정 기준을 만족하는 행만 남긴다. `df["avg_score"] >= 90` 은 학생별 True/False 값을 만들고, 이 값을 `df[...]` 에 넣으면 우수 학생만 남는다. 출석률과 과제 제출률 조건도 같은 방식으로 처리한다. 개수와 예시를 함께 출력해야 필터가 의도대로 작동했는지 확인할 수 있다.

**예상 핵심값**

| 조건 | 인원 |
|---|---:|
| 평균 90점 이상 | 107 |
| 평균 75점 미만 | 26 |
| 출석률 0.8 미만 | 73 |
| 과제 제출률 0.6 미만 | 46 |

---

## 문제 5 정답 — 정렬로 상위/하위 학생 보기

In [ ]:
top10_avg = df.sort_values("avg_score", ascending=False).head(10)
bottom10_avg = df.sort_values("avg_score", ascending=True).head(10)

cols = ["student_id", "grade", "class_name", "math", "english", "science", "avg_score", "attendance_rate", "homework_rate"]

print("평균 점수 상위 10명:")
print(top10_avg[cols])
print("평균 점수 하위 10명:")
print(bottom10_avg[cols])

top_student = top10_avg.iloc[0]
print("평균 1위:", top_student["student_id"], top_student["grade"], top_student["class_name"], f"{top_student['avg_score']:.2f}")
print("하위 10명 평균 출석률:", f"{bottom10_avg['attendance_rate'].mean():.2f}")
print("하위 10명 평균 과제 제출률:", f"{bottom10_avg['homework_rate'].mean():.2f}")

### 왜 이 코드가 정답인지

정렬은 상위/하위 학생을 빠르게 확인하는 방법이다. 내림차순으로 정렬하면 높은 점수가 위에 오고, 오름차순으로 정렬하면 낮은 점수가 위에 온다. 정렬 후 첫 행은 1위 또는 최하위 학생이므로 `iloc[0]` 으로 꺼낼 수 있다. 하위 10명의 출석률과 과제 제출률을 함께 보면 낮은 성적이 다른 지표와 연결되는지 살펴볼 수 있다.

**예상 핵심값**

평균 점수 1위는 `S0439` 이며 평균은 99.00점이다.

---

## 문제 6 정답 — 복합 조건으로 집중 관리 대상 찾기

In [ ]:
support_candidates = df[
    (df["avg_score"] < 75)
    | (df["attendance_rate"] < 0.8)
    | (df["homework_rate"] < 0.6)
]

excellent_candidates = df[
    (df["avg_score"] >= 90)
    & (df["attendance_rate"] >= 0.9)
    & (df["homework_rate"] >= 0.85)
]

print("지원 후보 수:", len(support_candidates))
print("우수 후보 수:", len(excellent_candidates))
print("지원 후보 낮은 평균 순:")
print(support_candidates.sort_values("avg_score").head(10)[cols])
print("우수 후보 높은 평균 순:")
print(excellent_candidates.sort_values("avg_score", ascending=False).head(10)[cols])

### 왜 이 코드가 정답인지

지원 후보는 세 조건 중 하나라도 해당하면 포함되므로 OR 연산자 `|` 를 사용한다. 우수 후보는 평균, 출석, 과제 조건을 모두 만족해야 하므로 AND 연산자 `&` 를 사용한다. pandas 조건식은 각 조건을 괄호로 감싸야 우선순위 오류를 피할 수 있다. 이 문제는 복합 조건을 안전하게 작성하는 것이 핵심이다.

**예상 핵심값**

```text
지원 후보 수: 122명
우수 후보 수: 27명
```

---

## 문제 7 정답 — 학년별·반별 요약

In [ ]:
df["support_flag"] = (
    (df["avg_score"] < 75)
    | (df["attendance_rate"] < 0.8)
    | (df["homework_rate"] < 0.6)
)

grade_summary = df.groupby("grade").agg(
    students=("student_id", "count"),
    avg_score=("avg_score", "mean"),
    avg_attendance=("attendance_rate", "mean"),
    avg_homework=("homework_rate", "mean"),
    support_count=("support_flag", "sum"),
)

class_summary = df.groupby("class_name").agg(
    students=("student_id", "count"),
    avg_score=("avg_score", "mean"),
    avg_attendance=("attendance_rate", "mean"),
    avg_homework=("homework_rate", "mean"),
    support_count=("support_flag", "sum"),
).sort_values("avg_score", ascending=False)

print("학년별 요약:")
print(grade_summary)
print("반별 요약:")
print(class_summary)
print("평균 점수 1위 학년:", grade_summary["avg_score"].idxmax())
print("평균 점수 1위 반:", class_summary["avg_score"].idxmax())

### 왜 이 코드가 정답인지

운영자는 학생 개별 목록뿐 아니라 학년과 반 단위의 흐름도 봐야 한다. `groupby` 로 학년별, 반별 학생 수와 평균 지표를 계산하면 어떤 그룹이 상대적으로 높은지 확인할 수 있다. 지원 후보 수를 함께 넣으면 평균 점수만 볼 때 놓치는 관리 부담도 볼 수 있다.

**예상 해석**

평균 점수는 3학년이 가장 높고, 반 기준으로는 A반이 가장 높다. 다만 지원 후보 수는 평균과 별도로 확인해야 한다.

---

## 문제 8 정답 — 운영용 핵심 점수 만들기

In [ ]:
df["core_score"] = (
    df["avg_score"] * 0.7
    + df["attendance_rate"] * 100 * 0.15
    + df["homework_rate"] * 100 * 0.15
)

print("핵심 점수 평균:", f"{df['core_score'].mean():.2f}")
print("핵심 점수 중앙값:", f"{df['core_score'].median():.2f}")
print("핵심 점수 최고:", f"{df['core_score'].max():.2f}")
print("핵심 점수 최저:", f"{df['core_score'].min():.2f}")

avg_top = df.loc[df["avg_score"].idxmax()]
core_top = df.loc[df["core_score"].idxmax()]
df["core_gap"] = df["core_score"] - df["avg_score"]

print("평균 점수 1위:", avg_top["student_id"], f"{avg_top['avg_score']:.2f}")
print("핵심 점수 1위:", core_top["student_id"], f"{core_top['core_score']:.2f}")
print("핵심 점수 상위 10명:")
print(df.sort_values("core_score", ascending=False).head(10)[["student_id", "avg_score", "attendance_rate", "homework_rate", "core_score"]])
print("평균 점수와 핵심 점수 차이가 큰 학생:")
print(df.reindex(df["core_gap"].abs().sort_values(ascending=False).head(5).index)[["student_id", "avg_score", "core_score", "core_gap"]])

### 왜 이 코드가 정답인지

`core_score` 는 성적 70%, 출석 15%, 과제 15%를 반영한 운영용 지표다. 출석률과 과제 제출률은 0~1 사이 값이므로 100을 곱해 점수와 같은 스케일로 맞춘다. 평균 점수 1위와 핵심 점수 1위가 다를 수 있는데, 이는 운영 지표가 성적 외의 성실도 요소를 반영하기 때문이다.

**예상 핵심값**

핵심 점수 1위는 `S0110` 이며, 출석률과 과제 제출률까지 높아 평균 점수 1위와 다를 수 있다.

---

## 문제 9 정답 — 상태 라벨 만들기

In [ ]:
df["risk_flag"] = df["support_flag"]
df["excellent_flag"] = (
    (df["avg_score"] >= 90)
    & (df["attendance_rate"] >= 0.9)
    & (df["homework_rate"] >= 0.85)
)

df["student_status"] = np.select(
    [df["excellent_flag"], df["risk_flag"]],
    ["excellent", "support"],
    default="stable",
)

print("상태별 학생 수:")
print(df["student_status"].value_counts())
print(df[["student_id", "avg_score", "attendance_rate", "homework_rate", "student_status"]].head(10))

### 왜 이 코드가 정답인지

운영 화면에서는 여러 숫자를 그대로 보여주는 것보다 상태 라벨이 필요할 때가 많다. `np.select` 는 조건 목록과 결과 목록을 받아 새 라벨을 만든다. 우수 조건을 먼저 둔 이유는 조건이 겹치는 경우 어떤 라벨을 우선할지 명확히 하기 위해서다. 이번 기준에서는 `excellent`, `support`, `stable` 세 상태를 만든다.

**예상 핵심값**

```text
stable: 331명
support: 122명
excellent: 27명
```

---

## 문제 10 정답 — 전체 순위와 반별 순위

In [ ]:
df["avg_rank"] = df["avg_score"].rank(ascending=False, method="min").astype(int)
df["core_rank"] = df["core_score"].rank(ascending=False, method="min").astype(int)
df["class_rank"] = df.groupby("class_name")["avg_score"].rank(ascending=False, method="min").astype(int)
df["rank_gap"] = df["avg_rank"] - df["core_rank"]

print("전체 평균 1위:")
print(df[df["avg_rank"] == 1][["student_id", "avg_score", "avg_rank"]])

print("핵심 점수 1위:")
print(df[df["core_rank"] == 1][["student_id", "core_score", "core_rank"]])

print("각 반 평균 1위:")
print(df[df["class_rank"] == 1][["class_name", "student_id", "avg_score", "class_rank"]].sort_values("class_name"))

print("평균 순위와 핵심 순위 차이가 큰 학생:")
print(df.reindex(df["rank_gap"].abs().sort_values(ascending=False).head(5).index)[["student_id", "avg_rank", "core_rank", "rank_gap"]])

### 왜 이 코드가 정답인지

`rank(ascending=False)` 는 높은 점수에 낮은 순위 번호를 준다. `method="min"` 은 동점자가 있으면 같은 최소 순위를 부여한다. 반별 순위는 `groupby("class_name")` 뒤에 `rank` 를 적용해 각 반 안에서만 순위를 계산한다. 평균 순위와 핵심 순위 차이를 보면 성적은 높지만 출석/과제가 약하거나, 반대로 성실도 덕분에 운영 점수가 오른 학생을 찾을 수 있다.

**채점 포인트**

- 전체 순위와 반별 순위를 구분했는가.
- 높은 점수가 1위가 되도록 `ascending=False` 를 사용했는가.
- 동점 처리 기준을 명확히 했는가.

---

## 문제 11 정답 — apply로 피드백 문장 만들기

In [ ]:
def make_feedback(row):
    if row["student_status"] == "excellent":
        return "성취도와 학습 습관이 모두 좋아 심화 과제를 제공할 수 있음"
    if row["student_status"] == "support":
        weak_points = []
        if row["avg_score"] < 75:
            weak_points.append("평균 점수")
        if row["attendance_rate"] < 0.8:
            weak_points.append("출석률")
        if row["homework_rate"] < 0.6:
            weak_points.append("과제 제출률")
        return ", ".join(weak_points) + " 확인 필요"
    return "현재 흐름은 안정적이며 유지 관리 중심으로 확인"

df["feedback"] = df.apply(make_feedback, axis=1)

print(df[["student_id", "avg_score", "attendance_rate", "homework_rate", "student_status", "feedback"]].head(10))

### 왜 이 코드가 정답인지

`apply(axis=1)` 은 각 행을 하나의 학생 기록으로 보고 함수를 적용한다. 조건이 복잡하고 문장을 만들어야 할 때는 단순 벡터 연산보다 함수가 읽기 쉽다. 우수 학생은 심화 과제, 지원 학생은 부족한 지표 확인, 안정 학생은 유지 관리로 문장을 나누면 운영자가 바로 참고할 수 있다.

**주의할 점**

이 피드백은 학부모에게 보내는 최종 문장이 아니라 내부 운영 참고 문장이다. 학생별 메시지로 쓰려면 말투와 개인정보 노출을 별도로 점검해야 한다.

---

## 문제 12 정답 — 학습 시간 구간 나누기

In [ ]:
df["study_level"] = pd.cut(
    df["study_hours"],
    bins=[0, 1.5, 3, 10],
    labels=["low", "mid", "high"],
    include_lowest=True,
)

study_summary = df.groupby("study_level", observed=True).agg(
    students=("student_id", "count"),
    avg_score=("avg_score", "mean"),
    avg_attendance=("attendance_rate", "mean"),
)

print("학습 시간 구간별 요약:")
print(study_summary)
print("평균 점수 1위 구간:", study_summary["avg_score"].idxmax())

### 왜 이 코드가 정답인지

`pd.cut` 은 연속형 숫자를 구간형 범주로 바꾼다. 학습 시간을 `low`, `mid`, `high` 로 나누면 구간별 학생 수와 평균 점수를 비교하기 쉽다. `observed=True` 는 실제 등장한 구간만 groupby 결과에 포함하게 한다. 구간화는 세부 숫자를 잃는 대신 해석이 쉬운 범주를 만든다.

**예상 핵심값**

평균 점수는 `mid` 구간이 가장 높게 나올 수 있다. 다만 차이가 작으므로 과장하지 않는다.

---

## 문제 13 정답 — 지표 간 상관관계 보기

In [ ]:
corr_cols = ["avg_score", "attendance_rate", "homework_rate", "study_hours"]
corr = df[corr_cols].corr()

print("상관계수:")
print(corr.round(3))

avg_corr = corr["avg_score"].drop("avg_score").abs().sort_values(ascending=False)
strongest_metric = avg_corr.index[0]
weakest_metric = avg_corr.index[-1]

print("avg_score와 가장 상관이 큰 지표:", strongest_metric, f"{corr.loc['avg_score', strongest_metric]:.3f}")
print("avg_score와 가장 상관이 약한 지표:", weakest_metric, f"{corr.loc['avg_score', weakest_metric]:.3f}")

# 상관관계는 같이 움직이는 정도이며, 원인과 결과를 바로 뜻하지 않는다.

### 왜 이 코드가 정답인지

상관계수는 두 숫자형 지표가 함께 움직이는 정도를 -1부터 1 사이 값으로 나타낸다. `avg_score` 와 다른 지표의 상관을 비교하면 성적과 출석, 과제, 학습 시간 중 어떤 지표가 더 함께 움직이는지 볼 수 있다. 절댓값으로 정렬하면 양의 상관과 음의 상관을 모두 크기 기준으로 비교할 수 있다.

**예상 핵심값**

`avg_score` 와 가장 상관이 큰 지표는 `homework_rate` 이고, `study_hours` 는 거의 상관이 없게 나올 수 있다.

---

## 문제 14 정답 — 운영 대시보드용 요약표 만들기

In [ ]:
total_students = len(df)
status_counts = df["student_status"].value_counts()
overall_metrics = {
    "avg_score": df["avg_score"].mean(),
    "avg_attendance": df["attendance_rate"].mean(),
    "avg_homework": df["homework_rate"].mean(),
}
status_by_class = pd.crosstab(df["class_name"], df["student_status"])

print("전체 학생 수:", total_students)
print("상태별 학생 수:")
print(status_counts)
print("전체 평균 지표:")
for key, value in overall_metrics.items():
    print(key, f"{value:.3f}")
print("반별 상태 분포:")
print(status_by_class)

print(f"지원 후보는 {status_counts.get('support', 0)}명이며, 우선 출석률과 과제 제출률이 낮은 학생부터 확인한다.")

### 왜 이 코드가 정답인지

대시보드용 요약은 개별 학생 목록보다 먼저 전체 규모와 위험 신호를 보여줘야 한다. 전체 학생 수, 상태별 학생 수, 평균 지표, 반별 상태 분포를 한 번에 보면 운영자가 어느 그룹을 먼저 볼지 결정할 수 있다. `pd.crosstab` 은 반과 상태의 조합별 인원을 세는 데 적합하다.

**채점 포인트**

- 전체 수와 상태별 수를 모두 출력했는가.
- 반별 상태 분포를 교차표로 만들었는가.
- 마지막 문장이 실제 숫자를 포함하는가.

---

## 문제 15 정답 — 성적표 feature engineering 결론

In [ ]:
avg_top = df.loc[df["avg_score"].idxmax()]
core_top = df.loc[df["core_score"].idxmax()]
support_count = (df["student_status"] == "support").sum()
excellent_count = (df["student_status"] == "excellent").sum()
most_support_class = status_by_class["support"].idxmax()
main_corr_metric = strongest_metric

print("평균 점수 1위:", avg_top["student_id"], f"{avg_top['avg_score']:.2f}")
print("핵심 점수 1위:", core_top["student_id"], f"{core_top['core_score']:.2f}")
print("지원 필요 학생 수:", support_count)
print("우수 학생 수:", excellent_count)
print("지원 학생이 가장 많은 반:", most_support_class)
print("성적과 가장 함께 움직인 지표:", main_corr_metric)

### 왜 이 코드가 정답인지

마지막 결론은 앞에서 만든 파생 지표를 다시 모아 운영 관점으로 해석하는 문제다. 평균 점수 1위와 핵심 점수 1위를 비교하면 성적 중심 지표와 운영 지표가 어떻게 다른지 볼 수 있다. 지원 필요 학생 수와 우수 학생 수는 운영 대상의 규모를 보여준다. 지원 학생이 많은 반과 상관 지표를 함께 보면 다음 행동을 정할 수 있다.

**결론 예시**

```text
평균 점수 1위는 S0439이지만, 출석률과 과제 제출률을 반영한 핵심 점수 1위는 S0110이다.
전체 480명 중 지원 필요 학생은 122명, 우수 학생은 27명으로 분류되었다.
지원 학생이 가장 많은 반은 A반이며, 성적과 가장 함께 움직인 지표는 과제 제출률이다.
다음 운영에서는 지원 학생 중 과제 제출률이 낮은 학생을 먼저 확인하고, 우수 학생에게는 심화 과제를 제공한다.
```

---

## 전체 채점 메모

| 구간 | 문제 | 핵심 개념 | 필수 통과 조건 |
|---|---|---|---|
| 구조 확인 | 1~2 | 로드, 기본 통계 | 480행 구조와 과목 통계 확인 |
| 파생 지표 | 3 | 평균, 총점 | `avg_score`, `total_score` 생성 |
| 필터/정렬 | 4~6 | 조건, 복합 조건, 정렬 | 우수/지원 학생군 계산 |
| 그룹 요약 | 7 | groupby | 학년별·반별 요약 생성 |
| 운영 지표 | 8~11 | core_score, 라벨, rank, apply | 상태와 피드백 문장 생성 |
| 해석 | 12~15 | cut, corr, crosstab, 결론 | 운영 가능한 결론 작성 |

## 학생 답안에서 자주 보는 패턴

| 패턴 | 의미 | 교사 피드백 |
|---|---|---|
| `axis=0` 으로 학생 평균 계산 | 행/열 방향 혼동 | 학생별 평균은 `axis=1` |
| 복합 조건에서 괄호 생략 | 연산 우선순위 오류 | 조건마다 괄호를 붙이게 함 |
| `and`, `or` 사용 | pandas Series 조건 처리 오류 | `&`, `|` 를 사용 |
| 출석률에 100을 곱하지 않음 | 스케일 불일치 | 점수와 같은 100점 기준으로 맞춤 |
| 순위가 낮은 점수부터 1위 | ascending 방향 오류 | 높은 점수가 1위면 `ascending=False` |
| 피드백 문장에 기준 없음 | 운영 활용 어려움 | 어떤 지표가 낮은지 포함 |

## 부분 점수 운영 기준

1. 문제 3의 평균 점수 계산이 틀리면 이후 모든 필터와 정렬이 흔들리므로 먼저 수정하게 한다.
2. 문제 6의 지원 후보 기준은 학생 상황에 따라 달라질 수 있지만, 이번 과제에서는 제시 기준을 그대로 적용해야 한다.
3. 문제 8의 핵심 점수 가중치는 다른 설계도 가능하다. 다만 과제에서는 70/15/15 기준을 지킨다.
4. 문제 11의 피드백 문장은 문구가 달라도 기준 지표가 반영되어 있으면 통과 가능하다.
5. 문제 15 결론에 숫자가 없거나, 코드 출력과 다른 내용을 쓰면 결론만 재작성하게 한다.

## 문제별 지도 질문

| 문제 | 학생에게 던질 질문 | 확인할 답 |
|---:|---|---|
| 1 | 이 데이터에서 한 행은 무엇을 의미하나요? | 학생 1명 |
| 2 | 과목별 평균은 왜 열 기준으로 계산하나요? | 열이 과목이기 때문 |
| 3 | 학생별 평균은 왜 `axis=1` 인가요? | 한 행의 세 과목을 평균 |
| 4 | 조건 필터 결과의 타입은 무엇인가요? | DataFrame |
| 5 | 정렬 후 1위 학생은 어떻게 꺼내나요? | `iloc[0]` 또는 `idxmax` |
| 6 | 지원 조건은 OR인가요 AND인가요? | 하나라도 해당하면 지원이므로 OR |
| 7 | groupby 결과는 어떤 질문에 답하나요? | 그룹별 평균/개수 |
| 8 | 출석률에 100을 곱한 이유는 무엇인가요? | 점수 스케일 맞춤 |
| 9 | 상태 라벨의 우선순위는 어떻게 되나요? | excellent 우선, support 다음 |
| 10 | 전체 순위와 반별 순위의 차이는 무엇인가요? | 비교 범위 |
| 11 | apply가 필요한 이유는 무엇인가요? | 행별 복합 문장 생성 |
| 12 | 구간화의 장단점은 무엇인가요? | 해석 쉬움, 세부 정보 손실 |
| 13 | 상관관계와 인과관계는 어떻게 다른가요? | 같이 움직임과 원인 구분 |
| 14 | 대시보드 첫 화면에 필요한 숫자는 무엇인가요? | 전체 수, 상태별 수, 핵심 평균 |
| 15 | 결론에 반드시 들어갈 것은 무엇인가요? | 숫자와 다음 행동 |

## 재실행 확인 순서

1. 런타임을 새로 시작한다.
2. 환경 셀부터 문제 15까지 순서대로 실행한다.
3. `avg_score`, `core_score`, `student_status`, `avg_rank`, `feedback` 열이 생겼는지 확인한다.
4. 상태별 학생 수가 전체 480명과 일치하는지 확인한다.
5. 결론 셀이 코드 출력과 같은 숫자를 사용하는지 확인한다.

정답 코드와 학생 코드가 달라도 같은 파생 지표를 만들고 같은 기준으로 해석하면 통과로 본다. 다만 운영용 지표는 기준이 조금만 달라도 결과가 바뀌므로, 기준을 주석과 결론에 분명히 남겨야 한다.